In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.fact_ventas")

In [0]:
spark.sql(
    """
    create table if not exists workspace.gold.fact_ventas(
        id_producto bigint,
        id_representante bigint,
        id_date bigint,
        unidades bigint,
        importe_venta decimal(12,2),
        importe_costo decimal(12,2),
        margen decimal(12,2)
    )
   
    """
)

In [0]:
from pyspark.sql import functions as F
df_ventas=spark.table("workspace.silver.tbl_ventas_detalle")

df_dim_date=spark.table("workspace.gold.dim_date")
df_dim_representante=spark.table("workspace.gold.dim_representante")
df_dim_producto=spark.table("workspace.gold.dim_producto")

#unir dimensiones para reemplazar valores por sus claves
df_fact_ventas=(
    df_ventas.join(
        df_dim_date.select( "id_date",F.col("date").alias("fecha")),
        on="fecha",
        how="inner")
    .join(
        df_dim_representante.select("id_representante","representante"),
        on="representante",
        how="inner")
    .join(
        df_dim_producto.select("id_producto","codigo_producto","precio_venta","costo_venta"),
        on="codigo_producto",
        how="inner")
)




In [0]:
df_fact_ventas = (
    df_fact_ventas
    .withColumn("importe_venta",(F.col("unidades") * F.col("precio_venta")).cast("decimal(12,2)"))
    .withColumn("importe_costo",(F.col("unidades") * F.col("costo_venta")).cast("decimal(12,2)"))
    .withColumn("margen",(F.col("importe_venta") - F.col("importe_costo")).cast("decimal(12,2)"))  
    .select("id_producto",
            "id_representante",
            "id_date",
            "unidades",
            "importe_venta",
            "importe_costo",
            "margen")

)


In [0]:
display(df_fact_ventas)
df_fact_ventas

In [0]:
df_fact_ventas.write.format("delta")\
    .option("mergeShema","true")\
    .mode("overwrite")\
    .saveAsTable("workspace.gold.fact_ventas")

In [0]:
%sql
select *from workspace.gold.fact_ventas